In [ ]:
%%file train.py


import os
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.ops import sigmoid_focal_loss


from tqdm import tqdm
from rich.console import Console
from rich.panel import Panel
import evaluate
import wandb
import timm

from accelerate import Accelerator
from accelerate.utils import set_seed

from nvidia.dali import fn, types, pipeline_def
from nvidia.dali.plugin.pytorch import DALIGenericIterator, LastBatchPolicy



gradient_accumulation_steps = 1
accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)

device_id = accelerator.local_process_index
num_shards = accelerator.num_processes
shard_id = accelerator.local_process_index

console = Console()


checkpoint_dir = "checkpoint"
log_file = "metrics_log.txt"
training_state_path = os.path.join(checkpoint_dir, "training_state.pt")
best_loss_model_path = "best_val_loss_model.pth"
best_accuracy_model_path = "best_val_accuracy_model.pth"



seed = 42
set_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



task = "righteye" # "head" "mouth" "righteye"
task_type = ""
classification_type = "binary" if task != "head" else "multiclass"


ENV_PATHS = {
    "Kaggle": {
        "train_1": f"/kaggle/input/datasets/manith04/ddd-train1-{task}{task_type}",
        "train_2": f"/kaggle/input/datasets/manith04/ddd-train2-{task}{task_type}",
        "eval": f"/kaggle/input/datasets/manith04/ddd-eval-{task}{task_type}",
    },
    "Colab": {     # Version must be check
        "train_1": f"/root/.cache/kagglehub/datasets/manith04/ddd-train1-{task}{task_type}/versions/1/",
        "train_2": f"/root/.cache/kagglehub/datasets/manith04/ddd-train2-{task}{task_type}/versions/1/",
        "eval": f"/root/.cache/kagglehub/datasets/manith04/ddd-eval-{task}{task_type}/versions/1/",
    },
}



env = os.environ
env_name = "Kaggle" if "KAGGLE_URL_BASE" in env else "Colab" if "COLAB_GPU" in env else "Local"


dataset_roots = ENV_PATHS[env_name]
train_dir_1 = dataset_roots["train_1"]
train_dir_2 = dataset_roots["train_2"]
val_dir = dataset_roots["eval"]



def create_metrics():
    metrics = {
        "accuracy": evaluate.load("accuracy"),
        "precision": evaluate.load("precision"),
        "recall": evaluate.load("recall"),
        "f1": evaluate.load("f1"),
    }
    return metrics


def update_metrics(metrics, preds, labels):
    for metric in metrics.values():
        metric.add_batch(predictions=preds, references=labels)


def compute_metrics(metrics):
    results = {}
    results["accuracy"] = metrics["accuracy"].compute()["accuracy"]
    results["precision"] = metrics["precision"].compute(average="macro")["precision"]
    results["recall"] = metrics["recall"].compute(average="macro")["recall"]
    results["f1"] = metrics["f1"].compute(average="macro")["f1"]
    return results



class DDDSequenceIndex():

    def __init__(self, root_dir, time_window_in_seconds, frames_per_window, window_stride_by_seconds):
        self.root_dir = root_dir
        self.time_window_in_seconds = time_window_in_seconds
        self.frames_per_window = frames_per_window
        self.window_stride_by_seconds = window_stride_by_seconds

        self.class_counts = {}
        self.sequence_frame_paths = []
        self.sequence_labels = []

        self._build_sequences()

        for class_label in sorted(self.class_counts):
            accelerator.print(f"Class {class_label} sequences: {self.class_counts[class_label]}")
        accelerator.print(f"Total sequences: {len(self.sequence_frame_paths)}\n")



    def _build_sequences(self):

        for subject_entry in os.scandir(self.root_dir):
            for scenario_entry in os.scandir(subject_entry.path):

                scenario_name = scenario_entry.name
                video_fps = 15 if "night" in scenario_name.lower() else 30

                for video_type_entry in os.scandir(scenario_entry.path):
                    self._process_windows_in_video(video_type_entry.path, video_fps)



    def _process_windows_in_video(self, video_path, video_fps):

        video_frame_list = sorted(entry.name for entry in os.scandir(video_path))
        total_frames_in_video = len(video_frame_list)
        available_frames_per_window = round(self.time_window_in_seconds * video_fps)

        stride_between_windows_in_frames = max(1, round(self.window_stride_by_seconds * video_fps))
        total_windows_in_video = (total_frames_in_video - available_frames_per_window) // stride_between_windows_in_frames + 1
        frames_space_in_window = (available_frames_per_window - 1) / (self.frames_per_window - 1)

        selected_frames_indices_in_window = [
            round(frame_position * frames_space_in_window)
            for frame_position in range(self.frames_per_window)
        ]

        last_frame_offset = selected_frames_indices_in_window[-1]

        frame_paths = [os.path.join(video_path, name)
                      for name in video_frame_list]

        frame_labels = [int(name.rsplit("_", 1)[-1].split(".")[0])
                        for name in video_frame_list]

        append_paths = self.sequence_frame_paths.append
        append_label = self.sequence_labels.append
        class_counts = self.class_counts


        # Create sequences
        for window_index in range(total_windows_in_video):
            window_start_frame = window_index * stride_between_windows_in_frames

            last_frame_index = (window_start_frame + last_frame_offset)
            if last_frame_index >= total_frames_in_video:
              break

            frame_indices = [window_start_frame + frame_offset for frame_offset in selected_frames_indices_in_window]

            # Remove Duplicates
            for frame_index in range(1, len(frame_indices)):
                if frame_indices[frame_index] <= frame_indices[frame_index - 1]:
                    frame_indices[frame_index] = frame_indices[frame_index - 1] + 1

            if frame_indices[-1] >= total_frames_in_video:
              break

            paths = [frame_paths[frame_index]
                    for frame_index in frame_indices]

            label = frame_labels[frame_indices[-1]]

            class_counts[label] = class_counts.get(label, 0) + 1

            append_paths(paths)
            append_label(label)







class WeightedSequenceBatchSource:

    def __init__(self, sequence_frame_paths, sequence_labels, batch_size,
        frames_per_window, shuffle=True, seed=42, shard_id=0, num_shards=1, drop_last=True):

        self.sequence_frame_paths = sequence_frame_paths
        self.batch_size = batch_size
        self.frames_per_window = frames_per_window
        self.shuffle = shuffle
        self.seed = seed
        self.shard_id = shard_id
        self.num_shards = num_shards
        self.drop_last = drop_last
        self.rng = random.Random(seed + shard_id)

        self.sequence_labels = [self._normalize_label(label) for label in sequence_labels]

        dataset_indices = list(range(len(self.sequence_labels)))
        self.shard_indices = dataset_indices[self.shard_id::self.num_shards]

        shard_labels = [self.sequence_labels[i] for i in self.shard_indices]
        labels_tensor = torch.tensor(shard_labels, dtype=torch.long)

        num_classes = int(labels_tensor.max().item()) + 1
        class_counts = torch.bincount(labels_tensor, minlength=num_classes).float()

        class_weights = torch.zeros_like(class_counts)
        class_weights[class_counts > 0] = 1.0 / class_counts[class_counts > 0]

        sample_weights = class_weights[labels_tensor]
        self.sample_weights = sample_weights.tolist()

        self.sequence_frame_paths_encoded = [
            [np.frombuffer(path.encode("utf-8"), dtype=np.uint8) for path in seq_paths]
            for seq_paths in self.sequence_frame_paths
        ]

        self.sequence_repeated_labels = [
            [np.array([label], dtype=np.float32) for _ in range(self.frames_per_window)]
            for label in self.sequence_labels
        ]

        self.num_samples = len(self.shard_indices)
        self.cursor = 0
        self.sampled_indices = []
        self.reset()



    def _normalize_label(self, label):
        if isinstance(label, torch.Tensor):
            label = label.item()
        elif isinstance(label, np.ndarray):
            label = label.item()
        elif isinstance(label, (list, tuple)):
            label = label[0]

        return int(label)



    def reset(self):

        if self.shuffle:
            self.sampled_indices = self.rng.choices( self.shard_indices, weights=self.sample_weights, k=self.num_samples)
        else:
            self.sampled_indices = list(self.shard_indices)

        self.cursor = 0


    def __len__(self):
        if self.drop_last:
            return len(self.sampled_indices) // self.batch_size
        return math.ceil(len(self.sampled_indices) / self.batch_size)


    def __call__(self):
        if self.cursor >= len(self.sampled_indices):
            raise StopIteration

        batch_indices = self.sampled_indices[self.cursor:self.cursor + self.batch_size]

        if self.drop_last and len(batch_indices) < self.batch_size:
            raise StopIteration

        self.cursor += len(batch_indices)

        flat_paths = [
            path
            for idx in batch_indices
            for path in self.sequence_frame_paths_encoded[idx]
        ]

        flat_labels = [
            label
            for idx in batch_indices
            for label in self.sequence_repeated_labels[idx]
        ]

        return flat_paths, flat_labels






class SequentialBatchSource():

    def __init__(self,sequence_frame_paths, sequence_labels, batch_size,  frames_per_window, drop_last=True, shard_id=0,num_shards=1):
        self.sequence_frame_paths = sequence_frame_paths
        self.sequence_labels = sequence_labels
        self.batch_size = batch_size
        self.frames_per_window = frames_per_window
        self.drop_last = drop_last
        self.shard_id = shard_id
        self.num_shards = num_shards

        dataset_window_indices = list(range(len(self.sequence_labels)))
        self.shard_window_indices = dataset_window_indices[self.shard_id::self.num_shards]

        if self.drop_last:
            usable = (len(self.shard_window_indices) // self.batch_size) * self.batch_size
            self.shard_window_indices = self.shard_window_indices[:usable]
            self.num_batches = len(self.shard_window_indices) // self.batch_size
        else:
            self.num_batches = math.ceil(len(self.shard_window_indices) / self.batch_size)

        self.sequence_frame_paths_encoded = [
            [np.frombuffer(path.encode("utf-8"), dtype=np.uint8) for path in seq_paths]
            for seq_paths in self.sequence_frame_paths
        ]

        self.sequence_repeated_labels = [
            [np.array([label], dtype=np.float32) for _ in range(self.frames_per_window)]
            for label in self.sequence_labels
        ]

        self.cursor = 0

    def reset(self):
        self.cursor = 0

    def __len__(self):
        return self.num_batches

    def __call__(self):
        if self.cursor >= len(self.shard_window_indices):
            raise StopIteration

        batch_indices = self.shard_window_indices[
            self.cursor:self.cursor + self.batch_size
        ]

        self.cursor += self.batch_size

        if len(batch_indices) < self.batch_size and self.drop_last:
            raise StopIteration

        flat_paths = [path for idx in batch_indices for path in self.sequence_frame_paths_encoded[idx]]
        flat_labels = [label for idx in batch_indices  for label in self.sequence_repeated_labels[idx]]

        return flat_paths, flat_labels




@pipeline_def
def train_sequence_pipeline(batch_source, image_height, image_width, frames_per_window):

    filepaths, labels = fn.external_source(source=batch_source, num_outputs=2, batch=True, parallel=False)

    encoded = fn.io.file.read(filepaths)
    images = fn.experimental.decoders.image(
        encoded,
        device="mixed",
        output_type=types.GRAY
    )

    images = fn.resize(images, resize_x=image_width, resize_y=image_height)

    mirror = fn.random.coin_flip(probability=0.5)
    images = fn.flip(images, horizontal=mirror)

    angle = fn.random.uniform(range=(-10.0, 10.0))
    images = fn.rotate(images, angle=angle, fill_value=0, keep_size=True)

    images = fn.crop_mirror_normalize(
        images,
        dtype=types.FLOAT,
        output_layout="CHW",
        mean=[127.5],
        std=[127.5],
    )

    labels = fn.cast(labels, dtype=types.FLOAT)
    return images, labels




@pipeline_def
def val_sequence_pipeline(batch_source, image_height, image_width, frames_per_window):
    filepaths, labels = fn.external_source(
        source=batch_source,
        num_outputs=2,
        batch=True,
        parallel=False
    )

    encoded = fn.io.file.read(filepaths)
    images = fn.experimental.decoders.image(
        encoded,
        device="mixed",
        output_type=types.GRAY
    )

    images = fn.resize(images, resize_x=image_width, resize_y=image_height)

    images = fn.crop_mirror_normalize(
        images,
        dtype=types.FLOAT,
        output_layout="CHW",
        mean=[127.5],
        std=[127.5],
    )

    labels = fn.cast(labels, dtype=types.FLOAT)
    return images, labels





class DALISequenceLoader():

    def __init__(self, iterator, batch_source, frames_per_window):
        self.iterator = iterator
        self.batch_source = batch_source
        self.frames_per_window = frames_per_window

    def __iter__(self):
        return iter(self.iterator)

    def __len__(self):
        return len(self.batch_source)

    def reset(self):
        self.batch_source.reset()
        self.iterator.reset()




def create_dali_dataloaders(accelerator, train_dirs, val_dir, batch_size, time_window_in_seconds,
    frames_per_window, window_stride_by_seconds, prefetch_queue_depth,
    image_height, image_width):

    device_id = accelerator.local_process_index
    shard_id = accelerator.local_process_index
    num_shards = accelerator.num_processes

    train_sequence_frame_paths = []
    train_sequence_labels = []
    combined_class_counts = {}

    for train_dir in train_dirs:
        train_index = DDDSequenceIndex(
            root_dir=train_dir,
            time_window_in_seconds=time_window_in_seconds,
            frames_per_window=frames_per_window,
            window_stride_by_seconds=window_stride_by_seconds,
        )

        train_sequence_frame_paths.extend(train_index.sequence_frame_paths)
        train_sequence_labels.extend(train_index.sequence_labels)

        for class_label, count in train_index.class_counts.items():
            combined_class_counts[class_label] = (
                combined_class_counts.get(class_label, 0) + count
            )

    val_index = DDDSequenceIndex(
        root_dir=val_dir,
        time_window_in_seconds=time_window_in_seconds,
        frames_per_window=frames_per_window,
        window_stride_by_seconds=window_stride_by_seconds,
    )

    train_source = WeightedSequenceBatchSource(
        sequence_frame_paths=train_sequence_frame_paths,
        sequence_labels=train_sequence_labels,
        batch_size=batch_size,
        frames_per_window=frames_per_window,
        shuffle=True,
        drop_last=True,
        seed=seed + shard_id,
        shard_id=shard_id,
        num_shards=num_shards,
    )

    val_source = SequentialBatchSource(
        sequence_frame_paths=val_index.sequence_frame_paths,
        sequence_labels=val_index.sequence_labels,
        batch_size=batch_size,
        frames_per_window=frames_per_window,
        drop_last=True,
        shard_id=shard_id,
        num_shards=num_shards,
    )

    train_pipe = train_sequence_pipeline(
        batch_source=train_source,
        image_height=image_height,
        image_width=image_width,
        frames_per_window=frames_per_window,
        batch_size=batch_size * frames_per_window,
        num_threads=4,
        device_id=device_id,
        seed=seed + shard_id,
        prefetch_queue_depth=prefetch_queue_depth,
    )

    val_pipe = val_sequence_pipeline(
        batch_source=val_source,
        image_height=image_height,
        image_width=image_width,
        frames_per_window=frames_per_window,
        batch_size=batch_size * frames_per_window,
        num_threads=4,
        device_id=device_id,
        seed=seed + shard_id,
        prefetch_queue_depth=prefetch_queue_depth,
    )

    train_pipe.build()
    val_pipe.build()

    train_iter = DALIGenericIterator(
        [train_pipe],
        output_map=["frames", "labels"],
        size=-1,
        auto_reset=False,
        prepare_first_batch=True,
    )

    val_iter = DALIGenericIterator(
        [val_pipe],
        output_map=["frames", "labels"],
        size=-1,
        auto_reset=False,
        prepare_first_batch=True,
    )

    train_loader = DALISequenceLoader(train_iter, train_source, frames_per_window)
    val_loader = DALISequenceLoader(val_iter, val_source, frames_per_window)

    return train_loader, val_loader, combined_class_counts




def get_criterion(loss_type, device, class_counts=None):
    loss_type = loss_type.lower()

    if loss_type == "focal_binary":
        alpha = -1
        gamma = 2.0

        def criterion(outputs, labels):
            return sigmoid_focal_loss(
                inputs=outputs,
                targets=labels.float(),
                alpha=alpha,
                gamma=gamma,
                reduction="mean"
            )

        return criterion

    elif loss_type == "bce":
        positive_weight_value = 1.0
        positive_weight = torch.tensor([positive_weight_value], dtype=torch.float32, device=device)
        return torch.nn.BCEWithLogitsLoss(pos_weight=positive_weight)

    elif loss_type == "ce":
        weight = None

        if class_counts is not None:
            class_counts_tensor = torch.tensor(class_counts, dtype=torch.float32, device=device)
            weight = class_counts_tensor.sum() / class_counts_tensor
            weight = weight / weight.sum() * len(class_counts)

        return torch.nn.CrossEntropyLoss(weight=weight)

    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")




def setup_training(accelerator, checkpoint_dir, training_state_path, log_file, device):

    best_val_loss = float("inf")
    best_val_accuracy = 0.0
    start_epoch = 0

    if os.path.isdir(checkpoint_dir) and os.listdir(checkpoint_dir):
        accelerator.print("⏭️ Loading checkpoint...")
        accelerator.load_state(checkpoint_dir)

        if os.path.exists(training_state_path):

            training_state = torch.load(training_state_path, map_location=device)
            start_epoch = training_state.get("epoch", 0)
            best_val_loss = training_state.get("best_val_loss", float("inf"))
            best_val_accuracy = training_state.get("best_val_accuracy", 0.0)
            accelerator.print(f"Resuming from epoch {start_epoch}")

        else:
            accelerator.print("🎯 No training state file found, starting from scratch.")

    if accelerator.is_main_process and start_epoch == 0:
        with open(log_file, "w") as f:
            f.write("epoch,train_loss,train_acc,train_precision,train_recall,train_f1," "val_loss,val_acc,val_precision,val_recall,val_f1\n")

    return start_epoch, best_val_loss, best_val_accuracy





def run_train_epoch( model, train_loader, optimizer, scheduler, accelerator, criterion,
    frames_per_window, classification_type,  epoch,  epochs,):
    model.train()

    if accelerator.is_main_process:
        print(f"\nEpoch {epoch + 1}/{epochs}")

    train_metrics = create_metrics()
    train_loss = 0.0
    num_train_batches = 0

    train_bar = tqdm(
        train_loader,
        desc="Training",
        disable=not accelerator.is_main_process
    )

    for batch_idx, data in enumerate(train_bar):
        batch = data[0]
        frames = batch["frames"]
        labels = batch["labels"]

        frames, labels = prepare_sequence_batch(
            frames=frames,
            labels=labels,
            frames_per_window=frames_per_window,
            classification_type=classification_type,
        )


        with accelerator.accumulate(model):
            outputs = model(frames)

            labels = labels.to(outputs.device, non_blocking=True)

            if classification_type == "binary":
                outputs = outputs.view(-1)
                labels = labels.float().view(-1)
            else:
                labels = labels.long()

            loss = criterion(outputs, labels)
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step(epoch + batch_idx / max(len(train_loader), 1))
                optimizer.zero_grad(set_to_none=True)

        gathered_loss = accelerator.gather_for_metrics(loss.detach().unsqueeze(0))
        train_loss += gathered_loss.mean().item()
        num_train_batches += 1

        preds = get_predictions(outputs.detach(), classification_type)

        if classification_type == "binary":
            labels_for_metrics = labels.long()
        else:
            labels_for_metrics = labels

        preds, labels_for_metrics = accelerator.gather_for_metrics((preds, labels_for_metrics))
        update_metrics(train_metrics, preds, labels_for_metrics)

    train_loader.reset()

    avg_train_loss = train_loss / max(num_train_batches, 1)
    train_results = compute_metrics(train_metrics)

    return avg_train_loss, train_results



def run_validation(model, val_loader, accelerator, criterion, frames_per_window, classification_type):
    model.eval()

    val_loss = 0.0
    num_val_batches = 0

    val_metrics = create_metrics()

    if classification_type == "binary":
        all_probs = []
        all_labels = []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", disable=not accelerator.is_main_process)

        for batch_idx, data in enumerate(val_bar):
            batch = data[0]
            frames = batch["frames"]
            labels = batch["labels"]

            frames, labels = prepare_sequence_batch(
                frames=frames,
                labels=labels,
                frames_per_window=frames_per_window,
                classification_type=classification_type,
            )

            outputs = model(frames)
            labels = labels.to(outputs.device, non_blocking=True)

            if classification_type == "binary":
                outputs = outputs.view(-1)
                labels = labels.float().view(-1)
            else:
                labels = labels.long()

            loss = criterion(outputs, labels)

            gathered_loss = accelerator.gather_for_metrics(loss.detach().unsqueeze(0))
            val_loss += gathered_loss.mean().item()
            num_val_batches += 1

            preds = get_predictions(outputs.detach(), classification_type, threshold=0.5)

            if classification_type == "binary":
                labels_for_metrics = labels.long()

                probs = torch.sigmoid(outputs.detach())
                probs_g, labels_g = accelerator.gather_for_metrics((probs, labels_for_metrics))

                all_probs.append(probs_g.cpu())
                all_labels.append(labels_g.cpu())
            else:
                labels_for_metrics = labels

            preds, labels_for_metrics = accelerator.gather_for_metrics((preds, labels_for_metrics))
            update_metrics(val_metrics, preds, labels_for_metrics)

    val_results = compute_metrics(val_metrics)
    avg_val_loss = val_loss / max(num_val_batches, 1)

    if classification_type == "binary":
        all_probs = torch.cat(all_probs)
        all_labels = torch.cat(all_labels)

        threshold_candidates = [round(x, 2) for x in torch.arange(0.10, 0.91, 0.01).tolist()]

        best_threshold = 0.5
        best_metrics = None
        best_accuracy = -1.0

        for threshold in threshold_candidates:
            temp_metrics = create_metrics()

            preds = (all_probs >= threshold).long()
            update_metrics(temp_metrics, preds, all_labels)

            temp_results = compute_metrics(temp_metrics)

            if temp_results["accuracy"] > best_accuracy:
                best_accuracy = temp_results["accuracy"]
                best_threshold = threshold
                best_metrics = temp_results

        val_results["best_threshold"] = best_threshold
        val_results["best_accuracy"] = best_metrics["accuracy"]
        val_results["best_precision"] = best_metrics["precision"]
        val_results["best_recall"] = best_metrics["recall"]
        val_results["best_f1"] = best_metrics["f1"]

    return avg_val_loss, val_results





def log_epoch_results(epoch, avg_train_loss, train_results, avg_val_loss, val_results, log_file, accelerator):

    if accelerator.is_main_process:
        print(
            f"Epoch {epoch + 1} - Train Loss: {avg_train_loss:.4f} | "
            f"Acc: {train_results['accuracy']:.4f} | "
            f"Precision: {train_results['precision']:.4f} | "
            f"Recall: {train_results['recall']:.4f} | "
            f"F1: {train_results['f1']:.4f}"
        )

        print(
            f"Epoch {epoch + 1} - Val   Loss: {avg_val_loss:.4f} | "
            f"Acc@0.5: {val_results['accuracy']:.4f} | "
            f"Precision@0.5: {val_results['precision']:.4f} | "
            f"Recall@0.5: {val_results['recall']:.4f} | "
            f"F1@0.5: {val_results['f1']:.4f}"
        )

        if "best_threshold" in val_results:
            print(
                f"Epoch {epoch + 1} - Val Best Threshold: {val_results['best_threshold']:.2f} | "
                f"Best Acc: {val_results['best_accuracy']:.4f} | "
                f"Best Precision: {val_results['best_precision']:.4f} | "
                f"Best Recall: {val_results['best_recall']:.4f} | "
                f"Best F1: {val_results['best_f1']:.4f}"
            )

        log_data = {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_accuracy": train_results["accuracy"],
            "train_precision": train_results["precision"],
            "train_recall": train_results["recall"],
            "train_f1": train_results["f1"],
            "val_loss": avg_val_loss,
            "val_accuracy": val_results["accuracy"],
            "val_precision": val_results["precision"],
            "val_recall": val_results["recall"],
            "val_f1": val_results["f1"],
        }

        if "best_threshold" in val_results:
            log_data["val_best_threshold"] = val_results["best_threshold"]
            log_data["val_best_accuracy"] = val_results["best_accuracy"]
            log_data["val_best_precision"] = val_results["best_precision"]
            log_data["val_best_recall"] = val_results["best_recall"]
            log_data["val_best_f1"] = val_results["best_f1"]

        wandb.log(log_data)

        with open(log_file, "a") as f:
            if "best_threshold" in val_results:
                f.write(
                    f"{epoch + 1},"
                    f"{avg_train_loss:.4f},{train_results['accuracy']:.4f},{train_results['precision']:.4f},{train_results['recall']:.4f},{train_results['f1']:.4f},"
                    f"{avg_val_loss:.4f},{val_results['accuracy']:.4f},{val_results['precision']:.4f},{val_results['recall']:.4f},{val_results['f1']:.4f},"
                    f"{val_results['best_threshold']:.2f},{val_results['best_accuracy']:.4f},{val_results['best_precision']:.4f},{val_results['best_recall']:.4f},{val_results['best_f1']:.4f}\n"
                )
            else:
                f.write(
                    f"{epoch + 1},"
                    f"{avg_train_loss:.4f},{train_results['accuracy']:.4f},{train_results['precision']:.4f},{train_results['recall']:.4f},{train_results['f1']:.4f},"
                    f"{avg_val_loss:.4f},{val_results['accuracy']:.4f},{val_results['precision']:.4f},{val_results['recall']:.4f},{val_results['f1']:.4f}\n"
                )




def save_best_models(model, accelerator, epoch, avg_val_loss, val_results,
    best_val_loss, best_val_accuracy, best_loss_model_path, best_accuracy_model_path):

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        if accelerator.is_main_process:
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": accelerator.get_state_dict(model),
                    "best_val_loss": best_val_loss,
                    "best_val_accuracy": best_val_accuracy,
                },
                best_loss_model_path,
            )

            artifact = wandb.Artifact(
                f"{experiment_no}-best-loss-model",
                type="model"
            )
            artifact.add_file(
                best_loss_model_path,
                name=f"{experiment_no}/best_loss_model/model.pth"
            )
            wandb.log_artifact(artifact)

            print("⭐ Best val loss model updated!")

    if val_results["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_results["accuracy"]

        if accelerator.is_main_process:
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": accelerator.get_state_dict(model),
                    "best_val_loss": best_val_loss,
                    "best_val_accuracy": best_val_accuracy,
                },
                best_accuracy_model_path,
            )

            artifact = wandb.Artifact(
                f"{experiment_no}-best-acc-model",
                type="model"
            )
            artifact.add_file(
                best_accuracy_model_path,
                name=f"{experiment_no}/best_acc_model/model.pth"
            )
            wandb.log_artifact(artifact)

            print("🔥 Best val accuracy model updated!")

    return best_val_loss, best_val_accuracy





def get_predictions(outputs, classification_type, threshold=0.5):
    if classification_type == "binary":
        outputs = outputs.view(-1)
        preds = (torch.sigmoid(outputs) >= threshold).long()
        return preds

    elif classification_type == "multiclass":
        preds = torch.argmax(outputs, dim=1)
        return preds

    else:
        raise ValueError("classification_type must be 'binary' or 'multiclass'")





def prepare_sequence_batch(frames, labels, frames_per_window, classification_type):
    flat_batch_size = frames.shape[0]
    real_batch_size = flat_batch_size // frames_per_window

    frames = frames.view(real_batch_size, frames_per_window, *frames.shape[1:])
    labels = labels.view(real_batch_size, frames_per_window, -1)[:, -1, 0]

    if classification_type == "binary":
        labels = labels.view(-1).float()
    elif classification_type == "multiclass":
        labels = labels.view(-1).long()
    else:
        raise ValueError("classification_type must be 'binary' or 'multiclass'")

    return frames, labels





def get_config(model_name):
    config = {
        "experiment_no": 26, # PLEASE CHANGE THIS EVERYTIME


        "epochs": 180,
        "batch_size": 64,
        "gradient_accumulation_steps": 1,
        "train_batch_strategy": "weighted_sampling_with_replacement",
        "image_height": 120,
        "image_width": 120,
        "prefetch_queue_depth": 256,
        "encoder_backbone_name": "mobilenetv3_small_100",

        "time_window_in_seconds": 4,
        "frames_per_window": 16,
        "window_stride_by_seconds": 2,


        "encoder_output_dim": 256,
        "encoder_dropout": 0.4,
        "temporal_dropout": 0.4,
        "finetune_start_layer": 0,

        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingWarmRestarts",
        "encoder_backbone_lr": 0.000_1,
        "encoder_head_lr":  0.001,
        "temporal_lr": 0.001,

        "classification_type": classification_type,   # "binary" or "multiclass"
        "loss_type": "bce",                # e.g. "bce" or "cross_entropy"
        "num_classes": 1,                  # 1 for binary, >1 for multiclass

        "weight_decay": 0.000_1,
        "eta_min": 0.000_001,
        "T_0": 10,
        "T_mult": 2,

    }

    if model_name == "lstm":
        config.update({
            "model_name": "lstm",
            "lstm_hidden_dim": 256,
            "lstm_num_layers": 2,
            "projection_head": False,
            "proj_size": 0,
        })

    elif model_name == "temporal_conv":
        config.update({
            "model_name": "temporal_conv",
            "tcn_hidden_dim": [512, 512, 512],
        })

    elif model_name == "transformer_encoder":
        config.update({
            "model_name": "transformer_encoder",
            "transformer_hidden_dim": 512,
            "transformer_num_layers": 4,
            "transformer_num_heads": 8,
        })

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return config




def create_encoder_backbone(config):
    backbone_name = config["encoder_backbone_name"]
    backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool="")
    return backbone



class CNNEncoder(nn.Module):

    def __init__(
        self, encoder_backbone_model, encoder_output_dim, encoder_dropout, finetune_start_layer=0):
        super().__init__()

        self.backbone = encoder_backbone_model
        self._convert_first_conv_to_grayscale()
        backbone_output_dim = self._get_backbone_output_dim()

        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(backbone_output_dim, encoder_output_dim),
            nn.LayerNorm(encoder_output_dim),
            nn.GELU(),
            nn.Dropout(encoder_dropout),
        )

        for param in self.backbone.parameters():
            param.requires_grad = False

        backbone_children = list(self.backbone.children())
        for child in backbone_children[finetune_start_layer:]:
            for param in child.parameters():
                param.requires_grad = True

    def _convert_first_conv_to_grayscale(self):
        first_conv_name = None
        first_conv_module = None

        for name, module in self.backbone.named_modules():
            if isinstance(module, nn.Conv2d):
                first_conv_name = name
                first_conv_module = module
                break

        if first_conv_module is None:
            raise ValueError("Could not find first Conv2d layer in backbone.")

        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=first_conv_module.out_channels,
            kernel_size=first_conv_module.kernel_size,
            stride=first_conv_module.stride,
            padding=first_conv_module.padding,
            dilation=first_conv_module.dilation,
            groups=first_conv_module.groups,
            bias=(first_conv_module.bias is not None),
            padding_mode=first_conv_module.padding_mode,
        )

        with torch.no_grad():
            if first_conv_module.weight.shape[1] == 3:
                new_conv.weight.copy_(first_conv_module.weight.mean(dim=1, keepdim=True))
            elif first_conv_module.weight.shape[1] == 1:
                new_conv.weight.copy_(first_conv_module.weight)
            else:
                raise ValueError(f"Expected first conv input channels to be 1 or 3, " f"but got {first_conv_module.weight.shape[1]}")

            if first_conv_module.bias is not None:
                new_conv.bias.copy_(first_conv_module.bias)

        self._set_module_by_name(first_conv_name, new_conv)

    def _set_module_by_name(self, module_name, new_module):
        parts = module_name.split(".")
        parent = self.backbone

        for part in parts[:-1]:
            if part.isdigit():
                parent = parent[int(part)]
            else:
                parent = getattr(parent, part)

        last_part = parts[-1]
        if last_part.isdigit():
            parent[int(last_part)] = new_module
        else:
            setattr(parent, last_part, new_module)


    def _get_backbone_output_dim(self):
        if hasattr(self.backbone, "num_features"):
            return self.backbone.num_features
        raise ValueError("Backbone does not expose num_features.")


    def forward(self, x):
        x = self.backbone.forward_features(x)

        if x.ndim == 4:
            x = self.pool(x)
        elif x.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected feature shape: {x.shape}")

        x = self.fc(x)
        return x








class LSTM(nn.Module):

    def __init__(self, encoder_backbone_model, encoder_output_dim, encoder_dropout,
                 lstm_hidden_dim, num_classes, lstm_num_layers, temporal_dropout, finetune_start_layer=0, proj_size=None):
        super().__init__()

        if proj_size >= lstm_hidden_dim:
            raise ValueError("proj_size must be smaller than lstm_hidden_dim")

        self.num_classes = num_classes
        self.is_binary = num_classes == 1
        self.use_projection = proj_size > 0

        self.encoder = CNNEncoder(
            encoder_backbone_model=encoder_backbone_model,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
            finetune_start_layer=finetune_start_layer,
        )

        self.lstm = nn.LSTM(
            input_size=encoder_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=temporal_dropout if lstm_num_layers > 1 else 0.0,
            proj_size=proj_size,
        )

        final_dim = proj_size if self.use_projection else lstm_hidden_dim
        self.classifier = nn.Linear(final_dim, num_classes)


    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)

        lstm_out, _ = self.lstm(features)
        last_timestep = lstm_out[:, -1, :]
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits



class CausalTemporalConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels, dilation, temporal_dropout, num_groups=8, kernel_size=3):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)

        self.norm1 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.norm2 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.norm3 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)

        self.activation = nn.GELU()
        self.dropout = nn.Dropout(temporal_dropout)

        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.residual(x)

        out = self.conv1(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm1(out)
        out = self.activation(out)
        out = self.dropout(out)

        out = self.conv2(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm2(out)
        out = self.activation(out)
        out = self.dropout(out)

        out = self.conv3(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm3(out)
        out = self.activation(out)
        out = self.dropout(out)

        return self.activation(out + residual)



class TemporalConv(nn.Module):

    def __init__(self, encoder_backbone_model, encoder_output_dim, encoder_dropout,
                        temporal_dropout, tcn_hidden_dim, num_classes=1, finetune_start_layer=0):
        super().__init__()

        if num_classes < 1:
            raise ValueError("num_classes must be >= 1")

        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_model=encoder_backbone_model,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
            finetune_start_layer=finetune_start_layer,
        )

        tcn_channels = tcn_hidden_dim
        num_tcn_blocks = len(tcn_channels)

        if num_tcn_blocks < 1:
            raise ValueError("num_tcn_blocks must be at least 1")

        tcn_dilations = [2 ** i for i in range(num_tcn_blocks)]
        blocks = []
        in_channels = encoder_output_dim

        for out_channels, dilation in zip(tcn_channels, tcn_dilations):
            blocks.append(
                CausalTemporalConvBlock(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    dilation=dilation,
                    temporal_dropout=temporal_dropout
                )
            )
            in_channels = out_channels

        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Linear(in_channels, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features.transpose(1, 2)

        tcn_out = self.tcn(features)
        last_timestep = tcn_out[:, :, -1]
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits



class TransformerEncoder(nn.Module):
    def __init__(self, encoder_backbone_model, encoder_output_dim, encoder_dropout, temporal_dropout,
                 num_heads, num_layers, dim_feedforward, max_len, num_classes, finetune_start_layer=0):
        super().__init__()

        if encoder_output_dim % num_heads != 0:
            raise ValueError("encoder_output_dim must be divisible by num_heads")

        if dim_feedforward is None:
            dim_feedforward = encoder_output_dim * 2

        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_model=encoder_backbone_model,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
            finetune_start_layer=finetune_start_layer
        )

        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, encoder_output_dim))

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=encoder_output_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=temporal_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)
        self.classifier = nn.Linear(encoder_output_dim, num_classes)



    def _generate_causal_mask(self, seq_len, device, dtype):
        mask = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=device, dtype=dtype),diagonal=1)
        return mask



    def forward(self, x):
        B, T, C, H, W = x.shape

        if T > self.pos_embedding.size(1):
            raise ValueError(f"Sequence length T={T} exceeds max_len={self.pos_embedding.size(1)}")

        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features + self.pos_embedding[:, :T, :]

        causal_mask = self._generate_causal_mask(
            seq_len=T,
            device=features.device,
            dtype=features.dtype
        )

        transformed = self.transformer(features, mask=causal_mask)
        last_timestep = transformed[:, -1, :]
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits





def build_model(config):
    encoder_backbone = create_encoder_backbone(config)

    if config["model_name"] == "lstm":
        model = LSTM(
            encoder_backbone_model=encoder_backbone,
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            lstm_hidden_dim=config["lstm_hidden_dim"],
            num_classes=config["num_classes"],
            lstm_num_layers=config["lstm_num_layers"],
            temporal_dropout=config["temporal_dropout"],
            finetune_start_layer=config["finetune_start_layer"],
            proj_size=config.get("proj_size", 0)

        )

    elif config["model_name"] == "temporal_conv":
        model = TemporalConv(
            encoder_backbone_model=encoder_backbone,
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            temporal_dropout=config["temporal_dropout"],
            tcn_hidden_dim=config["tcn_hidden_dim"],
            num_classes=config["num_classes"],
            finetune_start_layer=config["finetune_start_layer"],
        )

    elif config["model_name"] == "transformer_encoder":
        model = TransformerEncoder(
            encoder_backbone_model=encoder_backbone,
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            temporal_dropout=config["temporal_dropout"],
            num_heads=config["transformer_num_heads"],
            num_layers=config["transformer_num_layers"],
            dim_feedforward=config["transformer_hidden_dim"],
            max_len=config["frames_per_window"],
            num_classes=config["num_classes"],
            finetune_start_layer=config["finetune_start_layer"],
        )

    else:
        raise ValueError(f"Unknown model_name: {config['model_name']}")

    return model



def create_optimizer(model, config):
    temporal_lr = config["temporal_lr"]

    param_groups = [
        {"params": model.encoder.features.parameters(), "lr": config["encoder_backbone_lr"]},
        {"params": model.encoder.fc.parameters(), "lr": config["encoder_head_lr"]},
    ]

    if hasattr(model, "lstm"): param_groups.append({"params": model.lstm.parameters(), "lr": temporal_lr})
    if hasattr(model, "tcn"): param_groups.append({"params": model.tcn.parameters(), "lr": temporal_lr})
    if hasattr(model, "transformer"): param_groups.append({"params": model.transformer.parameters(), "lr": temporal_lr})
    if hasattr(model, "pos_embedding"): param_groups.append({"params": [model.pos_embedding], "lr": temporal_lr})

    param_groups.append({"params": model.classifier.parameters(), "lr": temporal_lr})
    return torch.optim.AdamW(param_groups, weight_decay=config["weight_decay"])




def create_scheduler(optimizer, config):
    if config["scheduler"] == "CosineAnnealingWarmRestarts":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=config["T_0"],
            T_mult=config["T_mult"],
            eta_min=config["eta_min"]
        )
        return scheduler

    raise ValueError(f"Unknown scheduler: {config['scheduler']}")




temporal_model = "lstm"  # "lstm", "temporal_conv", "transformer_encoder"
config = get_config(model_name=temporal_model)



experiment_no = config["experiment_no"]
if accelerator.is_main_process:
    console.print(Panel.fit(f"[bold yellow]Running Experiment:[/bold yellow]\n[bold green]{experiment_no}[/bold green]",border_style="bright_blue"))


train_loader, val_loader, class_counts = create_dali_dataloaders(
    accelerator=accelerator,
    train_dirs=[train_dir_1, train_dir_2],
    val_dir=val_dir,
    batch_size=config["batch_size"],
    time_window_in_seconds=config["time_window_in_seconds"],
    frames_per_window=config["frames_per_window"],
    window_stride_by_seconds=config["window_stride_by_seconds"],
    image_height=config["image_height"],
    image_width=config["image_width"],
    prefetch_queue_depth=config["prefetch_queue_depth"],
)



model = build_model(config)
optimizer = create_optimizer(model, config)
scheduler = create_scheduler(optimizer, config)


model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)
device = accelerator.device


criterion = get_criterion(
    loss_type=config["loss_type"],
    device=accelerator.device
)

start_epoch, best_val_loss, best_val_accuracy = setup_training(
    accelerator=accelerator,
    checkpoint_dir=checkpoint_dir,
    training_state_path=training_state_path,
    log_file=log_file,
    device=device
)

if accelerator.is_main_process:
    wandb.init(
        project=f"ddd-{task}",
        id=f"{experiment_no}",
        name=f"{experiment_no}",
        resume="allow",
        config=config,
        save_code=True

    )
    wandb.save("./train.ipynb")

optimizer.zero_grad(set_to_none=True)

for epoch in range(start_epoch, config["epochs"]):
    avg_train_loss, train_results = run_train_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        accelerator=accelerator,
        criterion=criterion,
        frames_per_window=config["frames_per_window"],
        classification_type=config["classification_type"],
        epoch=epoch,
        epochs=config["epochs"],
    )

    avg_val_loss, val_results = run_validation(
        model=model,
        val_loader=val_loader,
        accelerator=accelerator,
        criterion=criterion,
        frames_per_window=config["frames_per_window"],
        classification_type=config["classification_type"],
    )

    val_loader.reset()

    if accelerator.is_main_process:
        log_epoch_results(
            epoch=epoch,
            avg_train_loss=avg_train_loss,
            train_results=train_results,
            avg_val_loss=avg_val_loss,
            val_results=val_results,
            log_file=log_file,
            accelerator=accelerator
        )

        best_val_loss, best_val_accuracy = save_best_models(
            model=model,
            accelerator=accelerator,
            epoch=epoch,
            avg_val_loss=avg_val_loss,
            val_results=val_results,
            best_val_loss=best_val_loss,
            best_val_accuracy=best_val_accuracy,
            best_loss_model_path=best_loss_model_path,
            best_accuracy_model_path=best_accuracy_model_path,
        )

    accelerator.wait_for_everyone()
    accelerator.save_state(checkpoint_dir, safe_serialization=False)

    if accelerator.is_main_process:
        torch.save(
            {
                "epoch": epoch + 1,
                "best_val_loss": best_val_loss,
                "best_val_accuracy": best_val_accuracy,
            },
            training_state_path,
        )


if accelerator.is_main_process:
    wandb.finish()


In [ ]:
!uv pip install evaluate wandb nvidia-dali-cuda120 -q
!wandb login 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
!accelerate launch --num_processes=2  train.py